# Empirical Validation: ROC and Detection Delay

This notebook runs a Monte Carlo validation comparing CFAD to a raw-return CUSUM baseline on synthetic data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from cfad import detect
from cfad.utils import simulate_levy_returns

## Monte Carlo setup

We simulate normal series and series with a single jump, then compare CFAD ROC performance and detection delay.

In [ ]:
def simulate_series(n, jump=False):
    rng = np.random.default_rng()
    returns = rng.normal(loc=0.0, scale=0.01, size=n)
    if jump:
        returns[n // 2] += simulate_levy_returns(1, alpha=1.7, beta=0.0, scale=0.2)[0]
    return returns

def cusum_delay(scores, threshold=5.0, k=0.5):
    s = 0.0
    for t, value in enumerate(scores):
        z = value
        s = max(0.0, s + z - k)
        if s > threshold:
            return t
    return len(scores)

In [ ]:
n_trials = 100
n = 180
thresholds = np.linspace(3.0, 8.0, 11)
scores = []
labels = []
delays = []
for jump_flag in [False, True]:
    for _ in range(n_trials):
        returns = simulate_series(n, jump=jump_flag)
        report = detect(returns, window=60, h=5.0)
        scores.append(np.max(report.cusum_pos))
        labels.append(jump_flag)
        if jump_flag:
            delays.append(np.argmax(report.cusum_pos > report.threshold) if np.any(report.cusum_pos > report.threshold) else n)

scores = np.asarray(scores)
labels = np.asarray(labels)
delays = np.asarray(delays)
print(f'Completed {len(scores)} series, with {len(delays)} detected jumps')

## ROC curve

In [ ]:
fprs = []
tprs = []
for thresh in thresholds:
    preds = scores >= thresh
    tp = np.sum(preds & labels)
    fp = np.sum(preds & ~labels)
    fn = np.sum(~preds & labels)
    tn = np.sum(~preds & ~labels)
    fprs.append(fp / max(1, fp + tn))
    tprs.append(tp / max(1, tp + fn))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fprs, tprs, marker='o')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('CFAD ROC curve')
ax.grid(True, alpha=0.3)
figure_path = Path('paper/figures/04_roc_curve.png')
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=150)
print(f'Saved ROC curve to {figure_path}')

## Detection delay distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(delays, bins=20, color='tab:blue', alpha=0.7)
ax.set_xlabel('Detection delay (windows)')
ax.set_ylabel('Count')
ax.set_title('Empirical CFAD detection delay distribution')
ax.grid(True, alpha=0.3)
figure_path = Path('paper/figures/04_detection_delay.png')
fig.savefig(figure_path, dpi=150)
print(f'Saved detection delay figure to {figure_path}')